# 15b — Qwen3.5-4B size × limit × LoRA-1

**In one sentence:** on **unsloth/Qwen3.5-4B**, compare thinking OFF · ON · limits 512/1024/2048/4096 · and **LoRA-1** (full MBPP+ shortest-correct), on the same **234** test problems as the 2B thesis.

**Run order:** FIRST. Shared hour pot with the other size notebook: **≤150 hours combined**.

| | |
|---|---|
| Model profile | `qwen35_4b` |
| Drive folder | `MyDrive/stop-overthinking/results/4b/` |
| LoRA | **LoRA-1 only** (no LoRA-2) |
| Suggested hours for this run | ≤ 90 of the shared 150 |
| Dropped | "think briefly" (already explained on 2B) |

**Problem this solves:** the teacher asks "why only one model / why only 1024?".  
**Why:** same family, change only size + thinking limit + LoRA-1.  
**In:** 234 test problems + MBPP+ train for LoRA-1. **Out:** raw JSONL + graded CSV in this model's folder only.  
**Why this way:** 2B results stay untouched; each size has its own Colab and folder (DECISIONS #72).


## 1. Install

**Problem:** Colab starts empty. **Why:** Unsloth for LoRA-1, evalplus for grading.


In [ ]:
%%capture
!pip install -q unsloth
!pip install -q evalplus datasets flash-linear-attention


## 2. Config, GPU, Drive, shared hour pot

**Problem:** every path and the 150h cap must live in one place.  
**In:** hours left in the shared pot (read from Drive). **Out:** `T` folder for this model only.


In [ ]:
import os, sys, json, shutil, subprocess, time
from IPython import get_ipython

name, mem = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.strip().split(", ")
GPU, GPU_GB = name, float(mem) / 1024
PRODUCTION_OK = ("A100" in GPU or "H100" in GPU) and GPU_GB > 35
BATCH = 128 if GPU_GB > 35 else 16
print(f"GPU: {GPU} ({GPU_GB:.0f} GB) · batch {BATCH} · "
      + ("OK for the real run" if PRODUCTION_OK else "NOT an A100: only the smoke test will run"))

def rows(path):
    return sum(1 for l in open(path) if l.strip()) if os.path.exists(path) else 0

def sh(cmd):
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"FAILED (read the lines above): {cmd[:120]}")

from google.colab import drive
drive.mount("/content/drive")
REPO = "https://github.com/mahmudulhaquequdrati/stop-overthinking-thesis.git"
if not os.path.isdir("/content/thesis"):
    sh(f"git clone -q {REPO} /content/thesis")
else:
    sh("cd /content/thesis && git pull -q")
os.chdir("/content/thesis"); sys.path.insert(0, "/content/thesis/scripts")

# --- THIS RUN ONLY (do not write into the 2B thesis/ folder) ---
RUN_TAG   = "4b"
MODEL     = "qwen35_4b"
D         = "/content/drive/MyDrive/stop-overthinking/results"
T         = f"{D}/4b/raw"          # answers + LoRA for this size
SUMMARY   = f"{D}/4b/SUMMARY.md"
HOURS     = f"{D}/shared/hours_budget.json"      # shared 150h pot
LORA1     = f"{T}/lora/lora1"
WHEELS    = "/content/drive/MyDrive/stop-overthinking/wheels"
os.makedirs(T, exist_ok=True)
os.makedirs(LORA1, exist_ok=True)
os.makedirs(os.path.dirname(HOURS), exist_ok=True)
os.makedirs(WHEELS, exist_ok=True)

# Seed the shared pot on Drive if missing (first Colab to run).
if not os.path.exists(HOURS):
    json.dump(dict(cap_hours=150, used_hours=0.0, runs={},
                   note="shared pot for 0.8B + 4B; keep ≥50h of the 200 as buffer"),
              open(HOURS, "w"), indent=2)

MAXTOK = {"he": 4096, "lcb": 8192}
LIMITS = [512, 1024, 2048, 4096]          # the limit sweep (answers "why only 1024?")
SRC    = {"he": "humaneval", "lcb": "lcb"}
N      = {"he": 164, "lcb": 70}          # same 234 as notebook 14

# Ways: free options + LoRA-1. No "brief", no LoRA-2.
WAYS = {"off": ("thinking_off", None), "on": ("thinking_on", None)}
for L in LIMITS:
    WAYS[f"limit{L}"] = ("limit", None)   # think-budget set per call
WAYS["lora1"] = ("thinking_on", LORA1)

COST_H = {"smoke": 0.5, "A": 25.0, "B1": 12.0, "B2": 1.5, "B3": 5.0, "C": 30.0}

import hours_budget as hb
print(f"model={MODEL} · folder={T}")
print(f"shared hours left: {hb.left(HOURS):.1f} of 150")
print("Type the SHARED hours already used by the OTHER size run (0 if first):")
# Soft check only — the JSON on Drive is the real ledger.
_ = input("(press Enter to continue; ledger on Drive is authoritative) ")



## 3. Fast path (causal-conv1d), once per machine

**Problem:** without the fast kernel, answering is much slower. Same as notebook 14.


In [ ]:
import glob as _glob

def fast_path_ok():
    return subprocess.run([sys.executable, "-c", "import causal_conv1d"],
                          capture_output=True).returncode == 0

if fast_path_ok():
    print("fast path already importable")
else:
    found = _glob.glob(f"{WHEELS}/causal_conv1d*.whl")
    if found:
        sh(f"pip install -q {found[0]}")
    else:
        print("building/installing causal-conv1d (may be slow) ...")
        sh("pip install -q causal-conv1d || true")
    print("fast path OK" if fast_path_ok() else
          "WARNING: fast path missing — run will be slower but still correct")



## 4. Problems, overlap check, prompt switch check

**Problem:** same 234 test set; LoRA-1 trains on **all MBPP+ train problems** (100), full kept set.


In [ ]:
for c in ["python scripts/test_prompts.py",
           "python scripts/build_problem_set.py",
           "python scripts/mbpp_data.py",
           "python scripts/overlap_check.py"]:
    print(">", c); sh(c)
# Copy the exclude list into this run's folder (do not share with 2B Drive tree).
sh(f"cp data/overlap_exclude.json {T}/overlap-exclude.json")
print("problem set + overlap list ready")



## 5. Helpers (answer, grade, stage, hours)

**Problem:** same steps for every way; skip finished work; log hours into the shared pot.


In [ ]:
BG = []

def ans_path(way, ds, folder=None, prefix="test"):
    return f"{folder or T}/{prefix}-{way}-{ds}.jsonl"

def answer(way, ds, tries, folder=None, n=0, maxtok=None, think_budget=None,
           only_ids=None, prefix="test", stop_on_repeat=True, problems="data/problems.json"):
    out = ans_path(way, ds, folder, prefix)
    want = (len(json.load(open(only_ids))) if only_ids else (n or N[ds])) * tries
    if rows(out) >= want:
        print(f"  {way:<10}{ds:<4} complete ({rows(out)}) - skipped"); return out
    policy, adapter = WAYS[way]
    if way.startswith("limit") and think_budget is None:
        think_budget = int(way.replace("limit", ""))
    cmd = (f'python scripts/gen_colab.py --policy {policy} --label {way} --model {MODEL} '
           f'--problems {problems} --source {SRC[ds]} --samples {tries} '
           f'--dtype auto --batch {BATCH} --max-tokens {maxtok or MAXTOK[ds]} '
           f'--think-budget {think_budget or 1024} --out "{out}"')
    if stop_on_repeat: cmd += " --stop-on-repeat"
    if n: cmd += f" --n {n}"
    if adapter:
        if not os.path.exists(f"{adapter}/adapter_config.json"):
            raise RuntimeError(f"LoRA missing at {adapter} — train it before testing lora1")
        cmd += f' --adapter "{adapter}"'
    if only_ids: cmd += f' --only-ids "{only_ids}"'
    print(f"  {way:<10}{ds:<4} answering ...", flush=True)
    sh(cmd)
    return out

def graded_ok(ans):
    g = ans.replace(".jsonl", "-graded.csv")
    return os.path.exists(g) and rows(g) - 1 == rows(ans) and os.path.getmtime(g) >= os.path.getmtime(ans)

def grade(ans, ds, problems="data/problems.json", wait=False):
    for a, p in BG:
        if a == ans: p.wait()
    if graded_ok(ans): return
    if ds == "mbpp":
        script = "grade_mbpp.py"
    elif ds in ("he",):
        script = "grade_plus.py"
    else:
        script = "grade_lcb.py"
    p = subprocess.Popen([sys.executable, f"scripts/{script}", "--answers", ans, "--problems", problems],
                         stdout=open(ans + ".grade.log", "w"), stderr=subprocess.STDOUT)
    if wait: p.wait(); print("  graded", ans)
    else: BG.append((ans, p))

def wait_grading():
    for ans, p in BG:
        p.wait(); print("  graded", ans)
    BG.clear()

def free_ways():
    return ["off", "on"] + [f"limit{L}" for L in LIMITS]

def stage(name, ways, tries, cost_key=None):
    todo = [(w, ds) for w in ways for ds in SRC if rows(ans_path(w, ds)) < N[ds] * tries]
    if not todo:
        print(f"stage {name}: already complete"); return
    assert PRODUCTION_OK, "Production stages need the A100 (step 2)."
    cost = COST_H[cost_key or name]
    if not hb.gate(HOURS, RUN_TAG, name, cost): return
    hb.start_stage(HOURS, RUN_TAG, name)
    try:
        for w, ds in todo:
            grade(answer(w, ds, tries), ds)
    finally:
        hb.end_stage(HOURS, RUN_TAG, name)
print("helpers ready · free ways:", free_ways())



## 6. Smoke test (~0.5 h)

**Problem:** catch a broken switch / grader before burning hours.


In [ ]:
SM = f"{T}/smoke"
if os.path.exists(f"{SM}/PASSED"):
    print("smoke already passed - skipped")
elif hb.gate(HOURS, RUN_TAG, "smoke", COST_H["smoke"]):
    hb.start_stage(HOURS, RUN_TAG, "smoke")
    try:
        shutil.rmtree(SM, ignore_errors=True); os.makedirs(SM)
        for w in free_ways():
            for ds in SRC:
                grade(answer(w, ds, 1, folder=SM, n=2, maxtok=256, think_budget=128), ds, wait=True)
        for w in free_ways():
            for ds in SRC:
                r = [json.loads(l) for l in open(ans_path(w, ds, SM))]
                assert len(r) == 2, f"{w} {ds}: expected 2"
                if w == "off":
                    assert all(x["thinking_tokens"] == 0 for x in r), "OFF has thinking - STOP"
                else:
                    assert any(x["thinking_tokens"] > 0 for x in r), f"{w} counted 0 thinking - STOP"
                assert graded_ok(ans_path(w, ds, SM)), f"{w} {ds} not graded"
        open(f"{SM}/PASSED", "w").write("ok")
        print("SMOKE PASSED")
    finally:
        hb.end_stage(HOURS, RUN_TAG, "smoke")



## 7. Stage A — free ways on all 234 (try 1)

**Problem:** the teacher's table: OFF · ON · limits 512/1024/2048/4096.


In [ ]:
stage("A", free_ways(), tries=1, cost_key="A")
wait_grading()
print("stage A done (or skipped). Hours left:", round(hb.left(HOURS), 1))



## 8. Stage B — LoRA-1 only (all MBPP+ train problems, 100%)

**Problem:** rebuild the mini-thesis LoRA-1 recipe on **this** model. No LiveCodeBench train data.

```text
100 MBPP+ train → 4 tries ON → keep shortest correct → train LoRA on ALL kept → test on 234
```



In [ ]:
TR_ANS = f"{T}/train-mbpp-on.jsonl"
TRAIN  = f"{T}/train-set-lora1.jsonl"
EXCLUDE = f"{T}/overlap-exclude.json"

# B1: generate training answers (all MBPP+ train split, 4 tries)
if hb.gate(HOURS, RUN_TAG, "B1", COST_H["B1"]):
    hb.start_stage(HOURS, RUN_TAG, "B1")
    try:
        n_train = sum(1 for p in json.load(open("data/mbpp.json"))["problems"]
                      if p.get("split") == "train")
        want = n_train * 4
        print(f"MBPP+ train problems: {n_train} · want {want} answers")
        if rows(TR_ANS) < want:
            cmd = (f'python scripts/gen_colab.py --policy thinking_on --label on --model {MODEL} '
                   f'--problems data/mbpp.json --source mbpp --split train --samples 4 '
                   f'--dtype auto --batch {BATCH} --max-tokens 4096 --stop-on-repeat '
                   f'--out "{TR_ANS}"')
            print(cmd); sh(cmd)
        # grade with MBPP grader
        if not graded_ok(TR_ANS):
            sh(f'python scripts/grade_mbpp.py --answers "{TR_ANS}" --problems data/mbpp.json')
    finally:
        hb.end_stage(HOURS, RUN_TAG, "B1")

# B2: make train set + train LoRA-1 on ALL kept examples (fraction 1.0)
if os.path.exists(f"{LORA1}/adapter_config.json"):
    print("B2: LoRA-1 already trained - skipped")
elif hb.gate(HOURS, RUN_TAG, "B2", COST_H["B2"]):
    hb.start_stage(HOURS, RUN_TAG, "B2")
    try:
        sh(f'python scripts/make_train_set.py --answers "{TR_ANS}" --problems data/mbpp.json '
           f'--splits train --exclude "{EXCLUDE}" --out "{TRAIN}"')
        sh(f'python scripts/train_lora.py --train "{TRAIN}" --fraction 1.0 --epochs 3 '
           f'--model {MODEL} --out "{LORA1}"')
        assert os.path.exists(f"{LORA1}/adapter_config.json"), "LoRA-1 save failed"
    finally:
        hb.end_stage(HOURS, RUN_TAG, "B2")

# B3: test LoRA-1 on 234, try 1
if os.path.exists(f"{LORA1}/adapter_config.json"):
    stage("B3", ["lora1"], tries=1, cost_key="B3")
    wait_grading()
else:
    print("B3 skipped: no LoRA-1 adapter")



## 9. Stage C — second try (only if hours remain)

**Problem:** tighter error bars. Skip if the shared pot is tight.


In [ ]:
ways_c = free_ways() + (["lora1"] if os.path.exists(f"{LORA1}/adapter_config.json") else [])
stage("C", ways_c, tries=2, cost_key="C")
wait_grading()
print("hours left:", round(hb.left(HOURS), 1))



## 10. Finish grading + local summary table

**Problem:** write SUMMARY.md for this model; later join into ALL-RESULTS.


In [ ]:
wait_grading()
for way in free_ways() + (["lora1"] if os.path.exists(f"{LORA1}/adapter_config.json") else []):
    for ds in SRC:
        a = ans_path(way, ds)
        if rows(a) and not graded_ok(a):
            grade(a, ds, wait=True)

sh(f'python scripts/make_size_summary.py --dir "{T}" --run {RUN_TAG} --summary "{SUMMARY}"')
print("\n=== hours budget ===")
print(open(HOURS).read())
print(f"\nDone. Copy Drive results/{RUN_TAG}/ into the git repo when finished.")
print("Then locally: python scripts/make_all_results.py")

